# Cross-modal Learning: Adapting TERRA to H&E with CellBridge

Transfer **TERRA’s molecular representations** to **H&E images** with **CellBridge**. Using paired H&E–Xenium data, train **LoRA adapters** and an alignment head on frozen **UNI2-h** to match TERRA embeddings. Then fit a cell-type classifier and calibrate ontology-aware predictions with **Cell-Ontology-Fallback (COF)**.

**You will:**
1. Load pretrained CellBridge weights.
2. Optionally prepare TERRA targets for UNI2-h adaptation (HEST-1k).
3. Extract H&E features and fit a cell-type classifier (NuCLS).
5. Generate ontology-aware prediction sets with the COF module.
6. Optionally conduct qualitative analysis.

:::{note}
A **GPU is recommended** for image encoding and **TERRA requires an NVIDIA GPU** for optional target preparation.
:::

## 1. Setup

From the repository root, create and activate a **Python 3.11** environment:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
```
Install a [PyTorch build for your GPU](https://pytorch.org/get-started/locally/), then install CellBridge along with the notebook dependencies in the activated environment:

```bash
python -m pip install -e '.[models,preprocess,analysis]'
python -m pip install -r requirements/encoders.txt
python -m ipykernel install --user --name cellbridge --display-name 'CellBridge (Python 3.11)'
python -m jupyterlab notebooks/TUTORIAL.ipynb
```

For the optional TERRA section, also install `python -m pip install terra-st` following the [TERRA installation guide](https://terra-st.readthedocs.io/en/latest/installation.html).

Request access to [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h), accept the [HEST-1k dataset terms](https://huggingface.co/datasets/MahmoodLab/hest), and download the [CellBridge adapter](https://huggingface.co/ogutsevda/cellbridge).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from huggingface_hub import hf_hub_download

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/train/lora/cellbridge.yaml").is_file())
WORK = ROOT / "outputs" / "tutorial"
PAIRED = WORK / "data/train/hest"
NUCLS = WORK / "data/eval/nucls"
WORK.mkdir(parents=True, exist_ok=True)
print("Imports ready. Checking GPU availability...", flush=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREPARE_PAIRED = True

os.environ.update({
    "CELLBRIDGE_ROOT": str(ROOT),
    "CELLBRIDGE_DATA_ROOT": str(WORK / "data"),
    "CELLBRIDGE_OUTPUTS_ROOT": str(WORK / "results"),
    "CELLBRIDGE_CHECKPOINTS_ROOT": str(WORK / "checkpoints"),
})
print(f"Device: {DEVICE} | Outputs: {WORK}")

In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()

## 2. (Optional) Cross-modal alignment of TERRA and UNI2-h

If you set `PREPARE_PAIRED = False`, then you can skip this part, and directly use CellBridge in the next section.

If you would like to learn about the multi-modal pretraining step, please set `PREPARE_PAIRED = True`, and follow the example with two sections from HEST-1k below. You can also use custom paired datasets!

Below, we will download `NCBI876` (train) and `NCBI861` (validation), and extract TERRA cell embeddings. Check the [TERRA quickstart](https://terra-st.readthedocs.io/en/latest/notebooks/zero_shot_quickstart.html) for further details.

In [ ]:
# Download registered H&E and transcripts, then reconstruct raw cell counts.
if PREPARE_PAIRED:
    import yaml
    from huggingface_hub import snapshot_download

    HEST_RAW = WORK / "hest_raw"
    sample_ids = ["NCBI876", "NCBI861"]
    snapshot_download(
        "MahmoodLab/hest", repo_type="dataset", local_dir=str(HEST_RAW),
        allow_patterns=["HEST_v1_3_0.csv"] + [
            name for sid in sample_ids for name in (
                f"wsis/{sid}.tif", f"transcripts/{sid}_transcripts.parquet",
            )
        ],
    )
    hest_meta = pd.read_csv(HEST_RAW / "HEST_v1_3_0.csv", low_memory=False).set_index("id")
    PAIRED.mkdir(parents=True, exist_ok=True)
    manifest = pd.DataFrame([{
        "id": sid,
        "wsi": str(HEST_RAW / f"wsis/{sid}.tif"),
        "transcripts": str(HEST_RAW / f"transcripts/{sid}_transcripts.parquet"),
        "pixel_size_um": float(hest_meta.loc[sid, "pixel_size_um_estimated"]),
        "output_dir": str(PAIRED / sid),
    } for sid in sample_ids])
    manifest.to_csv(PAIRED / "manifest.csv", index=False)
    pd.DataFrame({
        "sample_id": sample_ids, "split": ["distill_train", "distill_val"],
    }).to_csv(PAIRED / "splits.csv", index=False)
    PREPROCESS_CONFIG = WORK / "preprocess.yaml"
    PREPROCESS_CONFIG.write_text(yaml.safe_dump({
        "output_root": str(PAIRED), "qv_threshold": 20,
    }))
    display(manifest[["id", "pixel_size_um"]])

if PREPARE_PAIRED:
    for sid in sample_ids:
        print(f"Reconstructing counts for {sid}...", flush=True)
        result = subprocess.run([
            sys.executable, "-m", "cellbridge.preprocess.02_quality_check_streaming",
            "--config", str(PREPROCESS_CONFIG), "--sample-id", sid,
        ], capture_output=True, text=True)

In [ ]:
# Match retained cell IDs to H&E centers before exporting the 384-dimensional TERRA targets.
if PREPARE_PAIRED:
    import anndata as ad
    from terra import download_pretrained, harmonize_adata, tokenize_adata, embed_dataset

    MAX_TEACHER_CELLS = 2_000  # Demo targets per section.
    TOKEN_CACHE = Path("/tmp") / f"cellbridge-{os.getuid()}" / "terra_tokens"
    TOKEN_CACHE.mkdir(parents=True, exist_ok=True)

    assert torch.cuda.is_available(), "Run TERRA in an NVIDIA GPU environment."
    terra_dir = Path(download_pretrained("Lotfollahi-lab/TERRA-112M"))

if PREPARE_PAIRED:
    for sid in sample_ids:
        sample_dir = PAIRED / sid
        adata = ad.read_h5ad(sample_dir / f"{sid}.h5ad")
        assert adata.obs_names.is_unique
        adata.obs["cell_id"] = adata.obs_names.astype(str)
        # Harmonize raw counts against TERRA's gene vocabulary.
        harmonized = harmonize_adata(
            adata,
            gene_mapping_dict_file_path=str(terra_dir / "ensembl_dictionary.pkl"),
            gene_occurrence_count_file_path=str(terra_dir / "gene_count_dictionary.pkl"),
            min_cells_per_gene=10, min_genes_per_cell=10,
        )
        # Build neighborhoods from the full section before selecting demo targets.
        tokens = tokenize_adata(
            adata=harmonized, model_folder_path=str(terra_dir),
            cache_directory_path=str(TOKEN_CACHE / sid), nproc=4,
            keep_in_memory=True,
        )
        tokens = tokens.shuffle(seed=0).select(range(min(MAX_TEACHER_CELLS, len(tokens))))
        print(f"{sid}: embedding {len(tokens):,} target cells", flush=True)
        # Run frozen TERRA, then export the selected cell-level alignment targets.
        teacher = embed_dataset(
            dataset=tokens, model_folder_path=str(terra_dir), batch_size=128, num_workers=4,
        )
        cell_ids = np.asarray(tokens["cell_id"], dtype=str)
        centers = harmonized.obs.loc[cell_ids, ["he_x", "he_y"]]
        features = np.asarray(teacher["cell_emb"], dtype=np.float32)
        assert len(np.unique(cell_ids)) == len(cell_ids)
        assert features.shape == (len(cell_ids), 384)
        assert np.isfinite(features).all() and np.isfinite(centers.to_numpy()).all()
        np.savez(
            sample_dir / f"{sid}_terra.npz", cell_ids=cell_ids, cell_emb=features,
            he_x=centers.he_x.to_numpy(), he_y=centers.he_y.to_numpy(),
        )
        print(f"{sid}: saved {features.shape} teacher features")

In [ ]:
# Preview a paired crop: 112 × 112 µm resized to 224 × 224 pixels.
if PREPARE_PAIRED:
    from cellbridge.train.lora.data import crop_cell, open_wsi

    sid = sample_ids[0]
    row = manifest.set_index("id").loc[sid]
    with np.load(PAIRED / sid / f"{sid}_terra.npz", allow_pickle=False) as saved:
        cx, cy = float(saved["he_x"][0]), float(saved["he_y"][0])
    wsi = open_wsi(row.wsi)
    try:
        display(crop_cell(wsi, cx, cy, row.pixel_size_um))
    finally:
        wsi.store.close()

In [ ]:
# Write an illustrative training config and print its command. This is only a simple example!
if PREPARE_PAIRED:
    import shlex
    from cellbridge.config import load_config

    train_cfg = load_config(ROOT / "configs/train/lora/cellbridge.yaml")
    train_cfg.update({
        "run_name": "tutorial_hest",
        "data_root": str(PAIRED), "splits_path": str(PAIRED / "splits.csv"),
        "output_dir": str(WORK / "training"),
        "checkpoint_dir": str(WORK / "checkpoints/tutorial_hest"),
        "max_cells_per_sample": 5000, "max_val_cells_per_sample": 2000,
    })
    train_cfg["training"].update({"max_steps": 20000, "eval_every": 1000, "patience": 5})
    TRAIN_CONFIG = WORK / "train.yaml"
    TRAIN_CONFIG.write_text(yaml.safe_dump(train_cfg, sort_keys=False))
    print(shlex.join(["cellbridge-train", "--config", str(TRAIN_CONFIG)]))

The **representative** training update is shown below:

```python
from cellbridge.train.lora.data import build_datasets
from cellbridge.train.lora.model import build_uni2_distiller, contrastive_loss
from cellbridge.train.lora.train import make_loader, make_optimizer

student = build_uni2_distiller(**train_cfg['model']).to(DEVICE)
train_data, val_data = build_datasets(train_cfg, student.eval_transforms)
loader = make_loader(train_data, train_cfg, training=True)
optimizer, _ = make_optimizer(student, train_cfg)
student.train()
for batch in loader:
    optimizer.zero_grad(set_to_none=True)
    image_z = student(batch['image'].to(DEVICE))['projected']
    target_z = student.project_terra(batch['terra'].to(DEVICE))
    loss = contrastive_loss(image_z, target_z, temperature=0.07)['loss_total']
    loss.backward()
    optimizer.step()
train_data.close()
val_data.close()
```

## 3. Download CellBridge and prepare NuCLS

Download the CellBridge adapter and UNI2-h weights. The extraction step loads the model when needed.

In [ ]:
LOCAL_CHECKPOINT = None
RELEASE = "1c334ee0783be5aaae93a5562522d73fc92ee6c1"
uni_weights = hf_hub_download("MahmoodLab/UNI2-h", filename="pytorch_model.bin")
checkpoint_path = Path(LOCAL_CHECKPOINT) if LOCAL_CHECKPOINT else Path(hf_hub_download(
    "ogutsevda/cellbridge", filename="cellbridge.pt", revision=RELEASE,
))
print(f"CellBridge checkpoint: {checkpoint_path}")
print(f"UNI2-h weights: {uni_weights}")

This command downloads the NuCLS dataset along with its cell-level annotations and prepares the fixed train/validation/test split!

In [ ]:
CONFIG = ROOT / "configs/inference/nucls.yaml"
RUN = WORK / "results/nucls"

subprocess.run(
    [sys.executable, "-m", "cellbridge.inference.prepare.nucls", str(NUCLS)],
    check=True,
)
metadata = pd.read_csv(NUCLS / "nucls_info.csv", dtype={"row_id": str})
display(metadata.groupby("split").size().rename("annotated_cells").to_frame())
with Image.open(NUCLS / "raw/rgb" / metadata.iloc[0]["image_name"]) as image:
    display(image.copy())

## 4. Check inputs and extract embeddings

We extract CellBridge features and run a simple linear probe. `check` validates the inputs and the checkpoint from the previous sections, whereas, `extract` loads UNI2-h in a separate process, applies that adapter, and saves one embedding per cell.

In [ ]:
def infer(stage):
    subprocess.run([
        sys.executable, "-m", "cellbridge.inference.run", stage,
        "--config", str(CONFIG), "--method", "cellbridge",
        "--arm", "lora_adapted", "--probe", "linear",
        "--checkpoint", str(checkpoint_path),
        "--output-dir", str(RUN), "--device", DEVICE,
    ], check=True)

infer("check")

⚠️ This step may take around 15 minutes!

In [ ]:
infer("extract")

In [ ]:
manifest = pd.read_csv(RUN / "embeddings/lora_adapted/manifest.csv")
print(f"Saved {manifest.n_cells.sum():,} cell embeddings across {len(manifest)} images.")
display(manifest[["sample_id", "n_cells", "n_features"]].head())

## 5. Evaluate the classifier

`evaluate` fits a logistic regression classifier using the saved embeddings and reports held-out test performance.

In [ ]:
infer("evaluate")
EVAL = RUN / "evaluation/lora_adapted/all/linear"
metrics = pd.read_csv(EVAL / "metrics.csv")
display(metrics[["macro_f1", "balanced_accuracy", "macro_auroc"]].round(3))

## 6. Calibrate and evaluate COF

`conformal` fits a separate classifier on training cells and uses validation cells for calibration.
It evaluates prediction sets on test cells at a target coverage of **90%**.
Ontology mapping and minimum class support can exclude cells from this stage.
CellBridge's **Cell-Ontology-Fallback (COF)** method corresponds to `branch_graph` below!

In [ ]:
infer("conformal")
COF = EVAL / "conformal_prediction"
cof_metrics = pd.read_csv(COF / "metrics.csv")
display(cof_metrics.loc[cof_metrics.conformal_method == "branch_graph", [
    "marginal_coverage", "mean_prediction_set_size", "singleton_rate",
]].round(3))

## 7. (Optional) Qualitative analysis

You can visualize the reliability scores after the COF module for the cell of your choice!

In [ ]:
with np.load(COF / "predictions.npz", allow_pickle=False) as saved:
    classes = saved["classes"].astype(str)
    sets = saved["branch_graph_sets"]
    predictions = pd.DataFrame({
        "row_id": saved["ids"].astype(str),
        "true_label": saved["y_true"].astype(str),
        "prediction_set": [", ".join(classes[mask]) for mask in sets],
        "set_size": sets.sum(axis=1),
    })

# Join by saved cell identity, not by CSV row order.
predictions = predictions.merge(
    metadata[["row_id", "image_name"]], on="row_id", validate="one_to_one",
)
display(predictions.head(10))
print(f"All results are saved under {RUN}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with np.load(COF / "predictions.npz", allow_pickle=False) as saved:
    top1 = pd.DataFrame({
        "row_id": saved["ids"].astype(str),
        "predicted_label": classes[saved["probabilities"].argmax(axis=1)],
    })
view = predictions.merge(top1, on="row_id", validate="one_to_one").merge(
    metadata[["row_id", "x", "y"]], on="row_id", validate="one_to_one",
)
view["reliability"] = 1 - view["set_size"] / len(classes)
image_name = view["image_name"].value_counts().index[0]  # Change to inspect another image.
cells = view.loc[view["image_name"] == image_name]
with Image.open(NUCLS / "raw/rgb" / image_name) as image:
    tissue = np.asarray(image.convert("RGB"))

fig, axes = plt.subplots(1, 4, figsize=(17, 5), layout="constrained")
for ax, title in zip(axes, ["H&E", "Reference labels", "Predicted labels (top-1)", "Reliability score"]):
    ax.imshow(tissue)
    ax.set_title(title)
    ax.axis("off")
colours = plt.get_cmap("tab20", len(classes))
for ax, column in zip(axes[1:3], ["true_label", "predicted_label"]):
    codes = pd.Categorical(cells[column], categories=classes).codes
    ax.scatter(cells.x, cells.y, c=codes, cmap=colours, vmin=-0.5,
               vmax=len(classes) - 0.5, s=24, edgecolors="black", linewidths=0.3)
nonempty = cells["set_size"] > 0
points = axes[3].scatter(cells.loc[nonempty, "x"], cells.loc[nonempty, "y"],
                         c=cells.loc[nonempty, "reliability"], cmap="RdYlGn",
                         vmin=0, vmax=1, s=24, edgecolors="black", linewidths=0.3)
if (~nonempty).any():
    axes[3].scatter(cells.loc[~nonempty, "x"], cells.loc[~nonempty, "y"],
                    c="gray", marker="x", s=30, label="Empty set")
    axes[3].legend(loc="lower right")
fig.colorbar(points, ax=axes[3], label="1 − set size / number of classes", shrink=0.7)
handles = [Line2D([], [], marker="o", linestyle="", color=colours(i), label=label)
           for i, label in enumerate(classes)]
fig.legend(handles=handles, loc="outside lower center", ncol=4, title="Cell types")
fig.suptitle(f"{image_name} · {len(cells)} evaluated test cells")
plt.show()